<a href="https://colab.research.google.com/github/bcarmoai/ML-and-AI/blob/main/HF_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [0]:
%pip install transformers torch accelerate

In [0]:
from huggingface_hub import login

login()

In [0]:
from transformers import pipeline

# 1. Initialize the local LLM pipeline
# device_map="auto" will automatically leverage Serverless GPUs if your session has them provisioned.
# We are using TinyLlama as it is a compact, highly capable instruct-model that fits easily into standard memory.
llm_pipeline = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto"
)

# 2. Define a function to interact with the local model
def query_offline_llm(user_prompt):
    # Format the prompt using a standard system/user chat template
    messages = [
        {"role": "system", "content": "You are a helpful and concise AI assistant."},
        {"role": "user", "content": user_prompt}
    ]

    # Generate the response locally without calling external APIs
    response = llm_pipeline(
        messages,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True
    )

    # Extract and return the assistant's reply from the output dictionary
    return response[0]['generated_text'][-1]['content']

# 3. Submit a prompt and get your offline answer
prompt = "Explain the Databricks concepts of Declarative Pipelines."
print(f"User Prompt: {prompt}\n")

answer = query_offline_llm(prompt)
print(f"Offline LLM Response:\n{answer}")

In [0]:
# 1. Define your new prompt
next_prompt = "Explain the Databricks concepts of Declarative Pipelines."
print(f"User Prompt: {next_prompt}\n")

# 2. Pass the prompt to the function defined in the previous cell
next_answer = query_offline_llm(next_prompt)

# 3. Print the output
print(f"Offline LLM Response:\n{next_answer}")

In [0]:
# 1. Redefine function with 2048 tokens to interact with the local model
def query_offline_llm_2048(user_prompt):
    # Format the prompt using a standard system/user chat template
    messages = [
        {"role": "system", "content": "You are a helpful and concise AI assistant."},
        {"role": "user", "content": user_prompt}
    ]

    # Generate the response locally without calling external APIs
    response = llm_pipeline(
        messages,
        max_new_tokens=2048,
        temperature=0.7,
        do_sample=True
    )

    # Extract and return the assistant's reply from the output dictionary
    return response[0]['generated_text'][-1]['content']

# 1. Define your new prompt
next_prompt = "Explain the difference between a Streaming Table and a Materialized View."
print(f"User Prompt: {next_prompt}\n")

# 2. Pass the prompt to the function defined in the previous cell
next_answer = query_offline_llm_2048(next_prompt)

# 3. Print the output
print(f"Offline LLM Response:\n{next_answer}")

In [0]:
import json
import re
from transformers.utils import get_json_schema

# 1. Define the ECL calculation function that will serve as the model's "tool"
#    Type hints + docstring are required so the schema can be auto-derived.
def calculate_monthly_ecl(pd: float, lgd: float, ead: float) -> str:
    """
    Calculate the monthly Expected Credit Loss (ECL) for a loan.

    Args:
        pd: Probability of Default as a decimal (e.g. 0.025 for 2.5%).
        lgd: Loss Given Default as a decimal (e.g. 0.40 for 40%).
        ead: Exposure at Default in the loan's currency (e.g. 1200000.0).

    Returns:
        The monthly Expected Credit Loss (pd * lgd * ead) as a string with exactly two decimal digits.
    """
    return f"{pd * lgd * ead:.2f}"

# 2. Convert the function into a JSON-schema tool definition the LLM can read
ecl_tools = [get_json_schema(calculate_monthly_ecl)]
print("Tool schema:")
print(json.dumps(ecl_tools, indent=2))

# 3. TinyLlama's chat template has no native 'tools' parameter, so we set the
#    tool up by injecting its schema into the system prompt and instructing
#    the model to answer with a single JSON tool-call object.
TOOL_SYSTEM_PROMPT = f"""You are a helpful assistant with access to the following tools:

{json.dumps(ecl_tools, indent=2)}

When the user's request requires a tool, respond ONLY with a single JSON object in this exact format and nothing else:
{{"name": "<tool name>", "arguments": {{<argument values>}}}}"""

def parse_tool_call(text):
    """Extract a JSON tool call from the model output, if present."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        call = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    return call if call.get("name") == "calculate_monthly_ecl" else None

# 4. Submit a prompt designed to trigger the tool call (retry a few times —
#    TinyLlama is small, so decoding is occasionally imperfect)
tool_messages = [
    {"role": "system", "content": TOOL_SYSTEM_PROMPT},
    {"role": "user", "content": "Calculate the monthly expected credit loss for a loan with a probability of default of 2.5%, a loss given default of 40%, and an exposure at default of $1,200,000."}
]

print(f"User Prompt: {tool_messages[1]['content']}\n")

tool_call = None
for attempt in range(3):
    response = llm_pipeline(tool_messages, max_new_tokens=256, temperature=0.1, do_sample=True)
    assistant_reply = response[0]["generated_text"][-1]["content"]
    print(f"--- Attempt {attempt + 1} model output ---\n{assistant_reply}\n")
    tool_call = parse_tool_call(assistant_reply)
    if tool_call:
        break

print(f"Parsed tool call: {tool_call}\n")

# 5. Execute the tool, then feed the result back for a final natural-language answer
if tool_call:
    args = {k: float(v) for k, v in tool_call["arguments"].items()}
    result = calculate_monthly_ecl(**args)
    print(f"Tool result: {result}\n")

    tool_messages.append({"role": "assistant", "content": assistant_reply})
    tool_messages.append({"role": "user", "content": f"The tool returned: {result}. Give the user a concise final answer, and also explain what ECL is."})
    follow_up = llm_pipeline(tool_messages, max_new_tokens=256, temperature=0.1, do_sample=True)
    print(f"Offline LLM Final Answer:\n{follow_up[0]['generated_text'][-1]['content']}")
else:
    print("The model did not produce a valid tool call.")

In [0]:
# Purpose: Demonstrate a complete local tool-calling loop using TinyLlama and a monthly ECL calculator.

import json
import re


# 1. Define the tool that Python is permitted to execute
def calculate_monthly_ecl(pd: float, lgd: float, ead: float) -> float:
    """Calculate monthly expected credit loss.

    Args:
        pd: Probability of default expressed as a decimal.
        lgd: Loss given default expressed as a decimal.
        ead: Exposure at default in pounds.

    Returns:
        Expected credit loss in pounds.
    """
    return pd * lgd * ead


# 2. Create an explicit registry of permitted tools
AVAILABLE_TOOLS = {
    "calculate_monthly_ecl": calculate_monthly_ecl
}


# 3. Extract a JSON object from TinyLlama's response
def extract_json_object(model_response: str) -> dict:
    """Extract and parse the first JSON object found in a model response."""

    cleaned_response = model_response.strip()

    # Remove Markdown code fences if the model includes them
    cleaned_response = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        cleaned_response,
        flags=re.IGNORECASE
    )

    # Find the first complete-looking JSON object
    json_match = re.search(r"\{.*\}", cleaned_response, flags=re.DOTALL)

    if not json_match:
        raise ValueError(
            f"The model did not return a JSON tool call.\n\n"
            f"Raw model response:\n{model_response}"
        )

    return json.loads(json_match.group(0))


# 4. Ask TinyLlama to select a tool
def request_tool_call(user_prompt: str) -> tuple[dict, str]:
    """Ask TinyLlama to choose a tool and provide its arguments."""

    system_prompt = """
You are a tool-selection assistant.

You have access to exactly one tool:

Tool name:
calculate_monthly_ecl

Description:
Calculates expected credit loss using the formula:
PD multiplied by LGD multiplied by EAD.

Arguments:
- pd: probability of default as a decimal number
- lgd: loss given default as a decimal number
- ead: exposure at default in pounds

If the user asks for an ECL calculation, respond with only one valid JSON
object in exactly this structure:

{
  "tool_name": "calculate_monthly_ecl",
  "arguments": {
    "pd": 0.03,
    "lgd": 0.40,
    "ead": 10000
  }
}

Do not calculate the answer yourself.
Do not include Markdown.
Do not include commentary before or after the JSON.
""".strip()

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    response = llm_pipeline(
        messages,
        max_new_tokens=256,
        do_sample=False,
        return_full_text=False
    )

    raw_response = response[0]["generated_text"]

    # Some Transformers versions may still return structured chat output
    if isinstance(raw_response, list):
        raw_response = raw_response[-1]["content"]

    tool_request = extract_json_object(raw_response)

    return tool_request, raw_response


# 5. Validate and execute the requested tool
def execute_tool_call(tool_request: dict) -> float:
    """Validate and execute a model-generated tool request."""

    tool_name = tool_request.get("tool_name")
    arguments = tool_request.get("arguments")

    if tool_name not in AVAILABLE_TOOLS:
        raise ValueError(f"Tool is not permitted: {tool_name}")

    if not isinstance(arguments, dict):
        raise ValueError("The tool arguments must be supplied as an object.")

    required_arguments = {"pd", "lgd", "ead"}

    missing_arguments = required_arguments - arguments.keys()
    unexpected_arguments = arguments.keys() - required_arguments

    if missing_arguments:
        raise ValueError(
            f"Missing tool arguments: {sorted(missing_arguments)}"
        )

    if unexpected_arguments:
        raise ValueError(
            f"Unexpected tool arguments: {sorted(unexpected_arguments)}"
        )

    validated_arguments = {
        "pd": float(arguments["pd"]),
        "lgd": float(arguments["lgd"]),
        "ead": float(arguments["ead"])
    }

    if not 0 <= validated_arguments["pd"] <= 1:
        raise ValueError("PD must be between 0 and 1.")

    if not 0 <= validated_arguments["lgd"] <= 1:
        raise ValueError("LGD must be between 0 and 1.")

    if validated_arguments["ead"] < 0:
        raise ValueError("EAD cannot be negative.")

    tool_function = AVAILABLE_TOOLS[tool_name]

    return tool_function(**validated_arguments)


# 6. Return the executed result to TinyLlama for a user-facing answer
def generate_final_answer(
    user_prompt: str,
    tool_request: dict,
    tool_result: float
) -> str:
    """Ask TinyLlama to explain the verified result."""

    arguments = tool_request["arguments"]

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful IFRS 9 assistant. "
                "The calculation has already been performed by a verified "
                "Python tool. Explain the result concisely. Do not recalculate "
                "or change the supplied values."
            )
        },
        {
            "role": "user",
            "content": user_prompt
        },
        {
            "role": "assistant",
            "content": (
                "I requested the calculate_monthly_ecl tool with "
                f"PD={arguments['pd']}, "
                f"LGD={arguments['lgd']} and "
                f"EAD={arguments['ead']}."
            )
        },
        {
            "role": "user",
            "content": (
                "The verified Python tool returned an expected credit loss "
                f"of £{tool_result:,.2f}. Provide the final answer."
            )
        }
    ]

    response = llm_pipeline(
        messages,
        max_new_tokens=256,
        do_sample=False,
        return_full_text=False
    )

    generated_text = response[0]["generated_text"]

    if isinstance(generated_text, list):
        generated_text = generated_text[-1]["content"]

    return generated_text.strip()


# 7. Run the complete tool-calling workflow
def query_tool_using_llm(user_prompt: str) -> str:
    """Select, execute and explain a tool call."""

    tool_request, raw_model_response = request_tool_call(user_prompt)

    print("Raw model tool request:")
    print(raw_model_response)

    print("\nParsed tool request:")
    print(json.dumps(tool_request, indent=2))

    tool_result = execute_tool_call(tool_request)

    print(f"\nVerified Python tool result: £{tool_result:,.2f}")

    return generate_final_answer(
        user_prompt=user_prompt,
        tool_request=tool_request,
        tool_result=tool_result
    )


# 8. Submit a test prompt
prompt = (
    "Calculate the monthly expected credit loss for an account with "
    "a probability of default of 3%, a loss given default of 40%, "
    "and an exposure at default of £10,000."
)

print(f"User prompt:\n{prompt}\n")

answer = query_tool_using_llm(prompt)

print(f"\nFinal model response:\n{answer}")

There are two fundamental flaws in the above cell:

Greedy decoding made failure permanent. do_sample=False is fully deterministic — I confirmed two consecutive runs produce byte-identical output. Once TinyLlama chose to answer in prose instead of JSON, re-running the cell could never succeed; you'd get the same failure every time. The working cell succeeds partly because it samples at low temperature and retries.

The prose tool description doesn't anchor TinyLlama to JSON. With that cell's prompt, even sampled decoding produced valid JSON 0/4 times. With the working cell's approach — embedding the tool's JSON schema (from get_json_schema) in the system prompt — sampled attempts succeeded consistently. A 1.1B model needs the JSON-shaped context to stay in JSON mode.

In [0]:
# Purpose: Native tool calling with a model whose chat template and training
# explicitly understand tool schemas — no prompt engineering, no JSON guessing.

import json
import re

from transformers import pipeline

# 1. Select the model: Qwen/Qwen2.5-1.5B-Instruct.
#    Its chat template natively renders the `tools` argument into Qwen's
#    Hermes-style tool-calling syntax, and the model is trained to answer
#    with the native tool-call protocol. At ~3GB it is the smallest tier
#    where this works reliably (the 0.5B tier drifts back to prose, exactly
#    like TinyLlama did).
qwen_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

# 2. Define the tool (the schema is auto-derived from type hints + docstring)
def calculate_monthly_ecl(pd: float, lgd: float, ead: float) -> str:
    """
    Calculate the monthly Expected Credit Loss (ECL) for a loan.

    Args:
        pd: Probability of Default as a decimal (e.g. 0.025 for 2.5%).
        lgd: Loss Given Default as a decimal (e.g. 0.40 for 40%).
        ead: Exposure at Default in the loan's currency (e.g. 10000.0).

    Returns:
        The monthly Expected Credit Loss (pd * lgd * ead) as a string with exactly two decimal digits.
    """
    return f"{pd * lgd * ead:.2f}"

AVAILABLE_TOOLS = {
    "calculate_monthly_ecl": calculate_monthly_ecl
}

# 3. Parse the model's native tool-call protocol.
#    The chat template injects the tool schema natively, but this
#    transformers version's pipeline does not auto-parse the response
#    marker back into structured tool_calls, so we extract the JSON payload
#    from the model's own documented protocol.
TOOL_CALL_PATTERN = re.compile(r"<tool_call\s*>?\s*(\{.*\})\s*</tool_call\s*>?", re.DOTALL)

def parse_native_tool_call(content: str) -> dict:
    """Parse a native tool call from the assistant message content."""
    match = TOOL_CALL_PATTERN.search(content)
    if match is None:
        raise ValueError(f"Model did not emit a native tool call: {content!r}")
    return json.loads(match.group(1))

# 4. Submit a prompt — the schema is injected by the chat template itself
messages = [
    {"role": "system", "content": "You are an IFRS 9 financial risk assistant. Use the calculate_monthly_ecl tool whenever an expected credit loss calculation is requested."},
    {"role": "user", "content": "Calculate the monthly expected credit loss for an account with a probability of default of 3%, a loss given default of 40%, and an exposure at default of £10,000. Additionally, provide insight into what the result means."}
]
print(f"User Prompt: {messages[1]['content']}\n")

response = qwen_pipeline(
    messages,
    tools=[calculate_monthly_ecl],
    max_new_tokens=256,
    do_sample=True,
    temperature=0.1
)
assistant_message = response[0]["generated_text"][-1]

# 5. Parse and execute the requested tool via the registry
tool_request = parse_native_tool_call(assistant_message["content"])
print(f"Parsed native tool call: {tool_request}")

tool_name = tool_request["name"]
if tool_name not in AVAILABLE_TOOLS:
    raise ValueError(f"Tool is not permitted: {tool_name}")

arguments = tool_request["arguments"]
result = AVAILABLE_TOOLS[tool_name](**{k: float(v) for k, v in arguments.items()})
print(f"Verified Python tool result: {result}\n")

# 6. Return the result to the model using its native 'tool' role, then get
#    the final user-facing answer
messages.append({"role": "assistant", "content": assistant_message["content"]})
messages.append({"role": "tool", "name": tool_name, "content": json.dumps({"monthly_ecl": result})})

final = qwen_pipeline(messages, max_new_tokens=256, do_sample=True, temperature=0.1)
print(f"Final LLM Answer:\n{final[0]['generated_text'][-1]['content']}")